# Differential Equations — Session 7
## Section 2.4: Exact Equations

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. connect total differentials with implicit solution families.
2. test exactness using $M_y=N_x$.
3. reconstruct a potential function $F(x,y)$.
4. apply an initial condition to an implicit solution.
5. interpret exact solutions as level curves.
6. find an integrating factor depending only on $x$ or only on $y$ in selected cases.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Topic |
|---:|---|
| 0–15 min | Total differentials |
| 15–30 min | Exactness criterion |
| 30–52 min | Reconstructing the potential |
| 52–67 min | Geometric level-curve interpretation |
| 67–82 min | Integrating factors for nonexact equations |
| 82–90 min | Practice and exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import erf
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def slope_field(f, xlim=(-3, 3), ylim=(-3, 3), density=21, title=None):
    x = np.linspace(*xlim, density)
    y = np.linspace(*ylim, density)
    X, Y = np.meshgrid(x, y)
    S = np.asarray(f(X, Y), dtype=float)
    S = np.nan_to_num(S, nan=0.0, posinf=20.0, neginf=-20.0)
    U = np.ones_like(S)
    length = np.sqrt(U**2 + S**2)
    plt.quiver(X, Y, U/length, S/length, angles="xy", pivot="mid")
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.xlabel("x")
    plt.ylabel("y")
    if title:
        plt.title(title)

def euler_method(f, x0, y0, h, n_steps):
    xs = np.empty(n_steps + 1)
    ys = np.empty(n_steps + 1)
    xs[0], ys[0] = x0, y0
    for n in range(n_steps):
        ys[n+1] = ys[n] + h*f(xs[n], ys[n])
        xs[n+1] = xs[n] + h
    return xs, ys

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 2.4-A — Exact differential equation

The differential equation

$$
M(x,y)\,dx+N(x,y)\,dy=0
$$

is **exact** on a region $R$ if there is a differentiable potential function $F(x,y)$ such that

$$
F_x=M,
\qquad
F_y=N.
$$

Its solutions are level curves

$$
F(x,y)=C.
$$

### Theorem 2.4-B — Exactness criterion

Suppose $M$, $N$, $M_y$, and $N_x$ are continuous on a simply connected rectangular region $R$. Then

$$
M\,dx+N\,dy
$$

is exact on $R$ if and only if

$$
M_y=N_x.
$$

### Potential reconstruction procedure

1. Integrate $M$ with respect to $x$:
   $$
   F(x,y)=\int M(x,y)\,dx+g(y).
   $$
2. Differentiate this expression with respect to $y$.
3. Match the result with $N(x,y)$ to determine $g'(y)$.
4. Integrate $g'(y)$ and write $F(x,y)=C$.

### Integrating-factor criteria

For a nonexact equation:

- If
  $$
  \frac{M_y-N_x}{N}
  $$
  depends only on $x$, then
  $$
  \mu(x)=\exp\left(\int\frac{M_y-N_x}{N}\,dx\right).
  $$

- If
  $$
  \frac{N_x-M_y}{M}
  $$
  depends only on $y$, then
  $$
  \mu(y)=\exp\left(\int\frac{N_x-M_y}{M}\,dy\right).
  $$

### Classroom Checkpoint — Exactness Test

For

$$
M(x,y)\,dx+N(x,y)\,dy=0,
$$

what derivative condition should be checked on a simply connected region?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Total differential

For a differentiable function $F(x,y)$,

$$
dF=F_x\,dx+F_y\,dy.
$$

If a curve satisfies

$$
F(x,y)=C,
$$

then along the curve,

$$
dF=0.
$$

Thus an equation

$$
M(x,y)\,dx+N(x,y)\,dy=0
$$

is exact when there is a potential function $F$ with

$$
F_x=M,\qquad F_y=N.
$$

## 2. Exactness criterion

On a suitable rectangular region, the equation is exact precisely when

$$
M_y=N_x.
$$

This is the equality of mixed partial derivatives.

## 3. Worked example

Solve

$$
(2xy-2)\,dx+(x^2+y^2)\,dy=0.
$$

Here

$$
M=2xy-2,\qquad N=x^2+y^2.
$$

Since

$$
M_y=2x=N_x,
$$

the equation is exact.

Integrate $F_x=M$ with respect to $x$:

$$
F=x^2y-2x+g(y).
$$

Differentiate with respect to $y$:

$$
F_y=x^2+g'(y).
$$

Matching $N=x^2+y^2$ gives

$$
g'(y)=y^2,
\qquad
g(y)=\frac{y^3}{3}.
$$

Therefore the implicit family is

$$
x^2y-2x+\frac{y^3}{3}=C.
$$

In [ ]:
# Verify with SymPy
x, y = sp.symbols("x y", real=True)
F = x**2*y - 2*x + y**3/3
display(sp.diff(F, x))
display(sp.diff(F, y))

## 4. Level curves and the gradient

The gradient

$$
\nabla F=(F_x,F_y)=(M,N)
$$

is perpendicular to every level curve $F=C$.

In [ ]:
x_vals = np.linspace(-2.5, 2.5, 300)
y_vals = np.linspace(-2.5, 2.5, 300)
X, Y = np.meshgrid(x_vals, y_vals)
Z = X**2*Y - 2*X + Y**3/3

levels = [-4, -2, -2/3, 0, 2, 4]
plt.contour(X, Y, Z, levels=levels)

# sparse gradient arrows
xs = np.linspace(-2, 2, 11)
ys = np.linspace(-2, 2, 11)
XX, YY = np.meshgrid(xs, ys)
M = 2*XX*YY - 2
N = XX**2 + YY**2
L = np.sqrt(M**2 + N**2)
L[L == 0] = 1
plt.quiver(XX, YY, M/L, N/L, angles="xy")
plt.scatter([1], [1], s=70, label="initial point")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Level curves and perpendicular gradient vectors")
plt.legend()
plt.show()

For the IVP passing through $(1,1)$,

$$
C=1-2+\frac13=-\frac23.
$$

The solution curve is the connected portion of the level curve $F=-2/3$ that contains the initial point.

## 5. A second exact equation

Consider

$$
(3x^2+2y)\,dx+(2x+4y^3)\,dy=0.
$$

It is exact, and the potential is

$$
F=x^3+2xy+y^4.
$$

Therefore

$$
x^3+2xy+y^4=C.
$$

In [ ]:
x_vals = np.linspace(-2, 2, 350)
y_vals = np.linspace(-1.8, 1.8, 350)
X, Y = np.meshgrid(x_vals, y_vals)
F2 = X**3 + 2*X*Y + Y**4

plt.contour(X, Y, F2, levels=np.linspace(-4, 6, 11))
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Implicit solution family $x^3+2xy+y^4=C$")
plt.show()

## 6. Making a nonexact equation exact

Consider

$$
y\,dx+(2x-y^2)\,dy=0.
$$

Here

$$
M=y,\qquad N=2x-y^2,
$$

so

$$
M_y=1,\qquad N_x=2.
$$

It is not exact.

Because

$$
\frac{N_x-M_y}{M}
=
\frac{2-1}{y}
=
\frac1y
$$

depends only on $y$, an integrating factor is

$$
\mu(y)=e^{\int 1/y\,dy}=y
$$

on intervals where $y$ has fixed sign.

After multiplication,

$$
y^2\,dx+(2xy-y^3)\,dy=0
$$

is exact. A potential is

$$
F=xy^2-\frac{y^4}{4},
$$

so the implicit family is

$$
xy^2-\frac{y^4}{4}=C.
$$

In [ ]:
x_vals = np.linspace(-3, 3, 350)
y_vals = np.linspace(-3, 3, 350)
X, Y = np.meshgrid(x_vals, y_vals)
Z = X*Y**2 - Y**4/4

plt.contour(X, Y, Z, levels=[-4, -2, -1, 0, 1, 2, 4])
plt.xlabel("x")
plt.ylabel("y")
plt.title("Level curves after applying an integrating factor")
plt.show()

## Interactive exploration — Select a level curve

For the potential

$$
F(x,y)=x^2y-2x+\frac{y^3}{3},
$$

changing $C$ selects a different implicit solution $F(x,y)=C$.

In [ ]:
def select_level(C=-2/3):
    x = np.linspace(-3, 3, 400)
    y = np.linspace(-3, 3, 400)
    X, Y = np.meshgrid(x, y)
    F = X**2*Y - 2*X + Y**3/3

    plt.contour(X, Y, F, levels=[C])
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(fr"Implicit solution $F(x,y)={C:.2f}$")
    plt.show()

if WIDGETS_AVAILABLE:
    interact(
        select_level,
        C=FloatSlider(min=-5, max=5, step=0.25, value=-2/3)
    )
else:
    select_level()

## Classroom Checkpoint — Exit Check

Determine whether

$$
(2x+y\cos x)\,dx+(\sin x+3y^2)\,dy=0
$$

is exact.

> Pause here. Let students commit to an answer before running the next cell.